🚀 Great. You've completed:

Day 1 → Data understanding + merge + popularity baseline
Day 2 → Weighted ratings + popularity filtering
Day 3 → User-based collaborative filtering
Day 4 → Item-based collaborative filtering
Day 5 → SVD / Matrix Factorization
Day 6 → Real Top-N recommendation engine
Day 7 → Content-based recommendation system

Now we combine multiple recommendation approaches into something closer to real systems.

🚀 Day 8 — Hybrid Recommendation System

Until now:

Collaborative filtering:

Users with similar behavior
↓
Recommend based on ratings

Content-based:

Movie features/genres
↓
Recommend similar items

Each has strengths and weaknesses.

Method	Strength	Weakness

Collaborative	Learns hidden preferences	Cold start problem
Content-based	Works for new items	Recommends very similar content


Real systems usually combine them.


---

🧠 Logic

Suppose:

Collaborative filtering says:

Titanic → 4.8
Avatar → 4.6
Batman → 4.2

Content-based says:

Titanic → 0.95
Notebook → 0.91
Pearl Harbor → 0.87

Combine both:

Final Score

=
Collaborative Score
+
Content Score

Then sort.


---

Step 1 — Create collaborative predictions

We already built this earlier:

user_id = 1

watched_movies = ratings[
    ratings['user_id']==user_id
]['movie_id'].tolist()

all_movies = ratings[
    'movie_id'
].unique()

predictions=[]

for movie in all_movies:

    if movie not in watched_movies:

        pred=model.predict(
            uid=user_id,
            iid=movie
        )

        predictions.append(
            (
                movie,
                pred.est
            )
        )


---

Step 2 — Convert predictions into DataFrame

collab_df = pd.DataFrame(
    predictions,
    columns=[
        'movie_id',
        'collab_score'
    ]
)

collab_df.head()

Example:

movie_id    collab_score

50          4.82
98          4.67
101         4.41


---

Step 3 — Create content score

Choose a movie from the user's history.

For now we simplify:

Take one movie the user liked.

liked_movie='Star Wars (1977)'

Find similarity scores:

movie_index = movies[
    movies['title']==liked_movie
].index[0]

content_scores = list(
    enumerate(
        similarity[movie_index]
    )
)

Convert to DataFrame:

content_df = pd.DataFrame(
    content_scores,
    columns=[
        'movie_index',
        'content_score'
    ]
)

content_df['movie_id']=movies[
    'movie_id'
]


---

Step 4 — Merge both scores

hybrid = pd.merge(
    collab_df,
    content_df,
    on='movie_id'
)


---

Step 5 — Create final score

We scale contributions:

hybrid[
    'final_score'
] = (

    0.7*
    hybrid[
        'collab_score'
    ]

    +

    0.3*
    hybrid[
        'content_score'
    ]
)

Weights mean:

70% collaborative
30% content


---

Step 6 — Sort recommendations

hybrid = hybrid.sort_values(
    'final_score',
    ascending=False
)

top10=hybrid.head(10)

for movie_id in top10[
    'movie_id'
]:

    movie_name=movies[
        movies[
            'movie_id'
        ]==movie_id
    ]['title'].values[0]

    print(movie_name)


---

Expected output

Example:

Empire Strikes Back
Return of the Jedi
Star Trek
Titanic
Shawshank Redemption


---

🧠 What happened?

Before:

Collaborative
↓
One recommendation source

or

Content-based
↓
One recommendation source

Now:

Collaborative score
+
Content score
↓
Combined ranking
↓
Better recommendation


---

Why real systems use hybrid methods

Netflix:

User behavior
+
watch history
+
genre preference
+
trending content

Amazon:

Purchase history
+
similar products
+
popularity

YouTube:

Watch time
+
click history
+
content similarity


---

🎯 Homework

1. Change weights:

Try:

0.5 / 0.5

and:

0.9 / 0.1

Check:

Do recommendations change?

Which feels better?



---

2. Think about this:

We used:

liked_movie='Star Wars'

Question:

What if user liked:

Star Wars
Titanic
Toy Story

How can we combine all liked movies instead of only one?

That idea leads toward user profiles and more production-like systems.



In [61]:
# Load dataset

from surprise import Dataset, Reader, SVD, accuracy

import pandas as pd

from surprise.model_selection import train_test_split

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.preprocessing import MinMaxScaler

import numpy as np

In [2]:
ratings_cols = [ 'user_id', 'movie_id', 'rating', 'timestamp' ]

ratings = pd.read_csv ( '../data/ml-100k/u.data', sep = '\t', names = ratings_cols )

ratings.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
movie_cols = [
    'movie_id',
    'title',
    'release_date',
    'video_release',
    'IMDb_URL',
    'unknown',
    'Action',
    'Adventure',
    'Animation',
    'Children',
    'Comedy',
    'Crime',
    'Documentary',
    'Drama',
    'Fantasy',
    'Film_Noir',
    'Horror',
    'Musical',
    'Mystery',
    'Romance',
    'SciFi',
    'Thriller',
    'War',
    'Western'
]

movies = pd.read_csv(
    '../data/ml-100k/u.item',
    sep='|',
    encoding='latin-1',
    header=None,
    names=movie_cols
)

movies.head()


,movie_id,title,release_date,video_release,IMDb_URL,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film_Noir,Horror,Musical,Mystery,Romance,SciFi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
# Create a movie features and combine them

genre_columns = movies.columns [5:]

#print(movies[genre_columns].iloc[0])

movies['genres'] = movies [ genre_columns ].apply ( lambda x : ' '.join ( x.index [ x == 1 ] ), axis = 1 )

movies [ [ 'title', 'genres' ] ].head()

,title,genres
0,Toy Story (1995),Animation Children Comedy
1,GoldenEye (1995),Action Adventure Thriller
2,Four Rooms (1995),Thriller
3,Get Shorty (1995),Action Comedy Drama
4,Copycat (1995),Crime Drama Thriller


In [5]:
# Convert text into numbers

# ActionComedyRomance -> [1, 1, 0, 0, 0, 1, 0, 0, .....]

cv = CountVectorizer ()

movie_vectors = cv.fit_transform ( movies [ 'genres' ] )


In [6]:
# Compute similarity

similarity = cosine_similarity ( movie_vectors )

print ( similarity )

[[1.         0.         0.         ... 0.         0.57735027 0.        ]
 [0.         1.         0.57735027 ... 0.         0.         0.        ]
 [0.         0.57735027 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.70710678]
 [0.57735027 0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.70710678 0.         1.        ]]


In [12]:
# Recommendation system

def recommend ( movie_name ):

    movie_index = movies [ movies [ 'title' ] == movie_name ].index [0]

    distances = list ( enumerate ( similarity [ movie_index ] ) )

    # print ( "Distances ->", distances )

    distances = sorted ( distances, key = lambda x : x [1], reverse = True ) 

    for i in distances [1:6]:

        if movies.iloc [ i [ 0 ] ][ 'title' ] != movie_name:

            print ( movies.iloc [ i[ 0 ] ][ 'title' ] )

print ( " --next : Batman Forever (1995) " )

recommend ( 'Batman Forever (1995)' )

print (" --next : Toy Story (1995)-- ")

recommend ( 'Toy Story (1995)' )

print (" --next : Titanic (1997)-- ")

recommend ( 'Titanic (1997)' )

print (" --next : Star Wars (1977)-- ")

recommend ( 'Star Wars (1977)' )

 --next : Batman Forever (1995) 
Batman Returns (1992)
Rumble in the Bronx (1995)
Batman & Robin (1997)
Three Musketeers, The (1993)
Cliffhanger (1993)
 --next : Toy Story (1995)-- 
Aladdin and the King of Thieves (1996)
Aladdin (1992)
Goofy Movie, A (1995)
Santa Clause, The (1994)
Home Alone (1990)
 --next : Titanic (1997)-- 
Man in the Iron Mask, The (1998)
Crying Game, The (1992)
First Knight (1995)
Postino, Il (1994)
 --next : Star Wars (1977)-- 
Return of the Jedi (1983)
Empire Strikes Back, The (1980)
Starship Troopers (1997)
African Queen, The (1951)
Stargate (1994)


In [13]:
# Prepare data for surprise

reader = Reader ( rating_scale = ( 1, 5 ) )  

data = Dataset.load_from_df ( ratings[ [ 'user_id', 'movie_id', 'rating' ] ], reader )

# Surprise expects ( user, item, rating )

# internally it builds -> matrices, latents vectors, embeddings 

In [14]:
# Train test split

trainset, testset = train_test_split ( data, test_size = 0.2, random_state = 42 )


In [15]:
# Train SVD model

model = SVD()

model.fit ( trainset )

# model is learning hidden user preferences, hidden movie characteristics, latent embeddings

# no direct cor relation

In [22]:
# Step 1 -> Collabrative filtering

user_id = 1

watched_movies = ratings [ ratings [ 'user_id' ] == user_id ][ 'movie_id' ].tolist()

all_movies = ratings [ 'movie_id' ].unique()

predictions = []

for movie in all_movies:

    if movie not in watched_movies:

        pred = model.predict ( uid = user_id, iid = movie )

        # print ( pred )

        predictions.append ( ( movie, pred.est ) )


In [27]:
# Step 2 Convert predictions into dataframe

collab_df = pd.DataFrame ( predictions, columns = [ 'movie_id', 'collab_score' ] )

collab_df.head()

,movie_id,collab_score
0,302,4.506166
1,377,2.910874
2,346,3.797958
3,474,4.125480
4,465,3.401516


In [23]:
# Step 3 - Creae a content score

liked_movie = 'Star Wars (1977)'

movie_index = movies [ movies [ 'title' ] == liked_movie ].index[0]

content_score = list ( enumerate ( similarity [ movie_index ] ) )


In [25]:
# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

print ( content_df )

      movie_index  content_score  movie_id
0               0       0.000000         1
1               1       0.516398         2
2               2       0.000000         3
3               3       0.258199         4
4               4       0.000000         5
...           ...            ...       ...
1677         1677       0.000000      1678
1678         1678       0.316228      1679
1679         1679       0.316228      1680
1680         1680       0.000000      1681
1681         1681       0.000000      1682

[1682 rows x 3 columns]


In [28]:
# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

In [29]:
# Create a final score

hybrid [ 'final_score' ] = ( 0.7 * hybrid [ 'collab_score' ] + 0.3 * hybrid [ 'content_score' ] )

In [36]:
# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    

Close Shave, A (1995)
Casablanca (1942)
Maltese Falcon, The (1941)
Manchurian Candidate, The (1962)
Titanic (1997)
Lawrence of Arabia (1962)
Persuasion (1995)
Affair to Remember, An (1957)
Bananas (1971)
Thin Man, The (1934)


# Home Work

In [41]:
# Change the weights -> 0.5  and 0.5

# Step 3 - Creae a content score

liked_movie = 'Star Wars (1977)'

movie_index = movies [ movies [ 'title' ] == liked_movie ].index[0]

content_score = list ( enumerate ( similarity [ movie_index ] ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

hybrid [ 'final_score' ] = ( 0.5 * hybrid [ 'collab_score' ] + 0.5 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.5 and 0.5 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.5 and 0.5 
Casablanca (1942)
Close Shave, A (1995)
Lawrence of Arabia (1962)
Titanic (1997)
African Queen, The (1951)
Crying Game, The (1992)
Persuasion (1995)
Affair to Remember, An (1957)
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
Forbidden Planet (1956)


In [42]:
# Change the weights -> 0.9  and 0.1

# Step 3 - Creae a content score

liked_movie = 'Star Wars (1977)'

movie_index = movies [ movies [ 'title' ] == liked_movie ].index[0]

content_score = list ( enumerate ( similarity [ movie_index ] ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

hybrid [ 'final_score' ] = ( 0.9 * hybrid [ 'collab_score' ] + 0.1 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.9 and 0.1 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.9 and 0.1 
Close Shave, A (1995)
Maltese Falcon, The (1941)
Manchurian Candidate, The (1962)
Casablanca (1942)
Thin Man, The (1934)
L.A. Confidential (1997)
Apt Pupil (1998)
Secrets & Lies (1996)
Titanic (1997)
Laura (1944)


In [46]:
# Change the weights -> 0.5  and 0.5 and liked movie

# Step 3 - Creae a content score

liked_movie = 'Batman (1989)'

movie_index = movies [ movies [ 'title' ] == liked_movie ].index[0]

content_score = list ( enumerate ( similarity [ movie_index ] ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

hybrid [ 'final_score' ] = ( 0.5 * hybrid [ 'collab_score' ] + 0.5 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.5 and 0.5 but for batman (1989) " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.5 and 0.5 but for batman (1989) 
Close Shave, A (1995)
Titanic (1997)
Secrets & Lies (1996)
All About Eve (1950)
Ben-Hur (1959)
Apt Pupil (1998)
Perfect World, A (1993)
Casablanca (1942)
To Kill a Mockingbird (1962)
Bonnie and Clyde (1967)


Nice — these outputs actually reveal something important. Your hybrid system is working mathematically, but it still has a design issue.

Look at this:

For Batman (1989) you got:

Close Shave
Titanic
All About Eve
Ben-Hur
Casablanca

That clearly doesn't match:

Batman
→ Action + Adventure + Superhero

So let's investigate.


---

🧠 Problem 1: Score scales are different

You're combining:

Collaborative score:

Range ≈ 1–5

Content similarity:

Range ≈ 0–1

But you did:

final_score = (
    0.7*collab_score
    +
    0.3*content_score
)

Suppose:

Movie A:

collab = 4.8
content = 0.2

Movie B:

collab = 3.5
content = 0.95

Final scores:

Movie A:

0.7(4.8)+0.3(0.2)

=3.42

Movie B:

0.7(3.5)+0.3(0.95)

=2.74

Movie A still wins easily because collaborative values are much larger.

Your content component has very little influence.


---

Fix → Normalize scores first

Before combining:

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

hybrid[
    ['collab_score']
] = scaler.fit_transform(
    hybrid[
        ['collab_score']
    ]
)

hybrid[
    ['content_score']
] = scaler.fit_transform(
    hybrid[
        ['content_score']
    ]
)

Now both become:

0 → 1

Example:

Movie A

collab=0.95
content=0.2

Movie B

collab=0.7
content=0.95

Now weights actually matter.


---

🧠 Problem 2: Using only one liked movie

You used:

liked_movie='Batman'

or:

liked_movie='Star Wars'

But real users like many movies:

Batman
Titanic
Toy Story

One movie cannot represent the whole user.


---

Instead:

Take several liked movies:

liked_movies=[
    'Batman (1989)',
    'Titanic (1997)',
    'Toy Story (1995)'
]

Compute similarity for each:

content_score=np.zeros(
    len(movies)
)

for movie in liked_movies:

    idx=movies[
        movies[
            'title'
        ]==movie
    ].index[0]

    content_score += similarity[
        idx
    ]

content_score=content_score/len(
    liked_movies
)

Now:

Batman similarity
+
Titanic similarity
+
Toy Story similarity
↓
Average user preference profile

This behaves more like:

Actual user taste

instead of:

User = Batman fan only


---

🧠 Big lesson from today:

You just discovered something common in production ML:

Combining models
≠
Simply adding outputs

You often need:

Scaling
+
normalization
+
proper weighting
+
user profile creation

before the combination starts behaving correctly.

This is exactly the kind of issue recommendation engineers debug in real systems.



In [65]:
# Change the weights -> 0.7  and 0.3

# Step 3 - Creae a content score

liked_movie = 'Batman (1989)'

movie_index = movies [ movies [ 'title' ] == liked_movie ].index[0]

content_score = list ( enumerate ( similarity [ movie_index ] ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

# Normalizing the score first

scaler = MinMaxScaler ()

hybrid [ 'collab_score' ] = scaler.fit_transform ( hybrid [ [ 'collab_score' ] ] )

hybrid [ 'content_score' ] = scaler.fit_transform ( hybrid [ [ 'content_score' ] ] )

hybrid [ 'final_score' ] = ( 0.7 * hybrid [ 'collab_score' ] + 0.3 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.5 and 0.5 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.5 and 0.5 
Ben-Hur (1959)
Titanic (1997)
Secrets & Lies (1996)
Perfect World, A (1993)
All About Eve (1950)
Menace II Society (1993)
Bonnie and Clyde (1967)
Once Were Warriors (1994)
To Kill a Mockingbird (1962)
Apt Pupil (1998)


In [66]:
#fix with multiple movies items

liked_movies = [ 'Batman (1989)', 'Titanic (1997)', 'Toy Story (1995)' ]

content_score = np.zeros ( len ( movies ) )

for movie in liked_movies:

    idx = movies [ movies [ 'title' ] == movie ].index [0]

    content_score += similarity [idx]

content_score = content_score / len (liked_movies)

content_score = list ( enumerate ( content_score ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

# Normalizing the score first

scaler = MinMaxScaler ()

hybrid [ 'collab_score' ] = scaler.fit_transform ( hybrid [ [ 'collab_score' ] ] )

hybrid [ 'content_score' ] = scaler.fit_transform ( hybrid [ [ 'content_score' ] ] )

hybrid [ 'final_score' ] = ( 0.7 * hybrid [ 'collab_score' ] + 0.3 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.9 and 0.1 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.9 and 0.1 
Titanic (1997)
Close Shave, A (1995)
Perfect World, A (1993)
Secrets & Lies (1996)
Crying Game, The (1992)
Mrs. Brown (Her Majesty, Mrs. Brown) (1997)
Casablanca (1942)
All About Eve (1950)
Ben-Hur (1959)
Leaving Las Vegas (1995)


IMPROVEMENTS

In [75]:
# SAME GENRE MOVIES -> tO BETTER UNDERSTAND WHEATHER ITS WORKING AND TO GET PRECISE OUTPUT

liked_movies = [
    'Batman (1989)',
    'Batman Returns (1992)',
    'Star Wars (1977)'
]


content_score = np.zeros ( len ( movies ) )

for movie in liked_movies:

    idx = movies [ movies [ 'title' ] == movie ].index [0]

    content_score += similarity [idx]

content_score = content_score / len (liked_movies)

content_score = list ( enumerate ( content_score ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

# Normalizing the score first

scaler = MinMaxScaler ()

hybrid [ 'collab_score' ] = scaler.fit_transform ( hybrid [ [ 'collab_score' ] ] )

hybrid [ 'content_score' ] = scaler.fit_transform ( hybrid [ [ 'content_score' ] ] )

hybrid [ 'final_score' ] = ( 0.1 * hybrid [ 'collab_score' ] + 0.9 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

print ( " for weights of 0.1 and 0.9 Similar genre ouput" )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


 for weights of 0.1 and 0.9 Similar genre ouput
Cliffhanger (1993)
Batman (1989)
Adventures of Robin Hood, The (1938)
Good Man in Africa, A (1994)
Highlander (1986)
Conan the Barbarian (1981)
Mortal Kombat (1995)
Ben-Hur (1959)
Waterworld (1995)
Kull the Conqueror (1997)


In [70]:

df = pd.merge ( ratings, movies, on = 'movie_id' )

In [77]:
# Using highest rated movies automatically

# add weighted sum for div instead of len () in content Score

liked_movies = df [ ( df [ 'user_id' ] == user_id ) & ( df [ 'rating' ] >= 4 ) ][ 'title' ].head( 5 ).tolist()

print ( liked_movies )

content_score = np.zeros ( len ( movies ) )

total_rating = 0

for movie in liked_movies:

    idx = movies [ movies [ 'title' ] == movie ].index [0]

    rating = df [ ( df [ 'user_id' ] == user_id ) & ( df [ 'title' ] == movie ) ][ 'rating' ].values [ 0 ]
    
    content_score += similarity [idx] * rating

    total_rating += rating # here we ar taking the weighted ratings

content_score = content_score / total_rating

content_score = list ( enumerate ( content_score ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

# Normalizing the score first

scaler = MinMaxScaler ()

hybrid [ 'collab_score' ] = scaler.fit_transform ( hybrid [ [ 'collab_score' ] ] )

hybrid [ 'content_score' ] = scaler.fit_transform ( hybrid [ [ 'content_score' ] ] )

hybrid [ 'final_score' ] = ( 0.7 * hybrid [ 'collab_score' ] + 0.3 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

# print ( " for weights of 0.9 and 0.1 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


['Three Colors: White (1994)', 'Desperado (1995)', 'Glengarry Glen Ross (1992)', 'Angels and Insects (1995)', 'Groundhog Day (1993)']
Mrs. Brown (Her Majesty, Mrs. Brown) (1997)
Leaving Las Vegas (1995)
Titanic (1997)
Casablanca (1942)
Secrets & Lies (1996)
All About Eve (1950)
Beautiful Thing (1996)
Apt Pupil (1998)
To Kill a Mockingbird (1962)
Shine (1996)


Yes — now it's behaving much more like we would expect 👍

Look at the first result set:

Cliffhanger
Highlander
Conan the Barbarian
Mortal Kombat
Waterworld
Kull the Conqueror

For:

Batman
Batman Returns
Star Wars

this is actually reasonable because the system sees:

Action
Adventure
SciFi
Fantasy

and says:

Find movies sharing these patterns

So your experiment proved something important:

0.1 collaborative
0.9 content
↓
Genre influence becomes visible

That validates:

✅ Content pipeline works
✅ Similarity matrix works
✅ Normalization works


---

Now for the second part:

content_score += similarity[idx] * rating

Good improvement 👍

You moved from:

All liked movies contribute equally

to:

Movie rated 5
→ stronger influence

Movie rated 4
→ weaker influence

That is closer to a real user profile.


---

One small improvement though:

You currently divide by:

content_score = content_score / len(liked_movies)

Suppose:

Movie A rating = 5
Movie B rating = 4
Movie C rating = 4
Movie D rating = 5
Movie E rating = 4

Weighted sum becomes:

5 + 4 + 4 + 5 + 4

But you're dividing only by:

5 movies

Not by total weight.

A more correct weighted average is:

total_rating = 0

for movie in liked_movies:

    idx = movies[
        movies['title']==movie
    ].index[0]

    rating = df[
        (df['user_id']==user_id)
        &
        (df['title']==movie)
    ]['rating'].values[0]

    content_score += (
        similarity[idx] * rating
    )

    total_rating += rating


content_score = (
    content_score /
    total_rating
)

Now:

Movie rated 5
↓
Has proportionally larger influence

instead of:

Movie rated 5
≈
Movie rated 4


---

🧠 You just moved through a realistic evolution:

Single liked movie
↓
Multiple liked movies
↓
Weighted profile
↓
Normalized profile

That's surprisingly close to the intuition behind how large recommendation systems construct user-interest vectors.

You’ve reached a stage where the code is no longer "toy example" territory.



In [80]:
# Using highest rated movies automatically

# add weighted sum for div instead of len () in content Score

liked_movies = df [ ( df [ 'user_id' ] == user_id ) & ( df [ 'rating' ] >= 4 ) ].sort_values ( 'rating', ascending = False )[ 'title' ]. head( 5 ).tolist()

print ( liked_movies )

content_score = np.zeros ( len ( movies ) )

total_rating = 0

for movie in liked_movies:

    idx = movies [ movies [ 'title' ] == movie ].index [0]

    rating = df [ ( df [ 'user_id' ] == user_id ) & ( df [ 'title' ] == movie ) ][ 'rating' ].values [ 0 ]
    
    content_score += similarity [idx] * rating

    total_rating += rating # here we ar taking the weighted ratings

content_score = content_score / total_rating

content_score = list ( enumerate ( content_score ) )

# Converting to data frame

content_df = pd.DataFrame ( content_score, columns = [ 'movie_index', 'content_score' ] )

content_df [ 'movie_id' ] = movies [ 'movie_id' ]

# print ( content_df )

# Merge both content and collab scores

hybrid = pd.merge ( collab_df, content_df, on = 'movie_id' )

# Create a final score

# Normalizing the score first

scaler = MinMaxScaler ()

hybrid [ 'collab_score' ] = scaler.fit_transform ( hybrid [ [ 'collab_score' ] ] )

hybrid [ 'content_score' ] = scaler.fit_transform ( hybrid [ [ 'content_score' ] ] )

hybrid [ 'final_score' ] = ( 0.7 * hybrid [ 'collab_score' ] + 0.3 * hybrid [ 'content_score' ] )

# Sort the recommendation

hybrid = hybrid.sort_values ( 'final_score', ascending = False )

top10 = hybrid.head ( 10 )

# print ( " for weights of 0.9 and 0.1 " )

for movie_id in top10 ['movie_id']:

    movie_name = movies [ movies [ 'movie_id' ] == movie_id ][ 'title' ].values [ 0 ]

    print ( movie_name )
    


['Sleeper (1973)', "Monty Python's Life of Brian (1979)", 'Henry V (1989)', 'Ridicule (1996)', 'Haunted World of Edward D. Wood Jr., The (1995)']
As Good As It Gets (1997)
Close Shave, A (1995)
Secrets & Lies (1996)
Swimming with Sharks (1995)
Rosencrantz and Guildenstern Are Dead (1990)
Wag the Dog (1997)
All About Eve (1950)
Cool Hand Luke (1967)
Bananas (1971)
Paths of Glory (1957)


This actually looks more coherent than the previous outputs 👀

Let's inspect the user's top-rated movies:

Sleeper
Monty Python's Life of Brian
Henry V
Ridicule
Haunted World of Edward D. Wood Jr.

Themes roughly become:

Comedy
Satire
Drama
Historical
Dark humor

Now look at recommendations:

As Good As It Gets
Swimming with Sharks
Wag the Dog
Bananas
Rosencrantz and Guildenstern Are Dead

Those also lean toward:

Comedy
Drama
Satire
Character-driven stories

So the flow is behaving more like:

Highest-rated movies
↓
Build user taste profile
↓
Weight by rating
↓
Create content signal
↓
Combine with collaborative signal
↓
Rank movies

rather than:

Random liked movies
↓
Mixed profile
↓
Noisy recommendations

You can also notice something interesting:

Close Shave
Secrets & Lies

still appear repeatedly.

That often happens because collaborative filtering (SVD) keeps assigning them high predicted scores for this user.

You can verify by printing:

hybrid[
    [
        'movie_id',
        'collab_score',
        'content_score',
        'final_score'
    ]
].sort_values(
    'final_score',
    ascending=False
).head(10)

You might see something like:

movie         collab   content   final

Close Shave    0.97      0.63      0.87
Secrets        0.93      0.58      0.82

That debugging step is valuable because now you're not just looking at movie names; you're looking at why a movie ranked highly.

You're moving from:

Does the model work?

to:

Why did the model make this recommendation?

That shift becomes important for explainable recommendation systems.



In [82]:
# Add movies to this hybrid and verify 

hybrid[
    [
        'movie_id',
        'collab_score',
        'content_score',
        'final_score'
    ]
].sort_values(
    'final_score',
    ascending=False
).head(10)


,movie_id,collab_score,content_score,final_score
418,316,0.761258,1.000000,0.832880
106,408,1.000000,0.408248,0.822474
329,285,0.838080,0.707107,0.798788
60,789,0.710610,1.000000,0.797427
585,652,0.705104,1.000000,0.793573
108,347,0.700925,1.000000,0.790647
639,606,0.822120,0.707107,0.787616
363,523,0.693120,1.000000,0.785184
675,502,0.815794,0.707107,0.783188
456,641,0.806166,0.707107,0.776449


Yes 🚀 Day 8 reached a good stopping point.

You covered much more than just "combine two models."

📌 Day 8 Summary — Hybrid Recommendation System

Goal

Build a recommendation system by combining:

Collaborative Filtering
+
Content-Based Filtering

instead of relying on a single method.


---

What we implemented

Step 1

Used SVD collaborative predictions:

pred=model.predict(
    uid=user_id,
    iid=movie
)

Generated:

movie_id → predicted rating


---

Step 2

Created content similarity:

similarity = cosine_similarity(
    movie_vectors
)

Initial approach:

liked_movie='Batman'

Limitation:

Single movie
≠
Entire user preference


---

Step 3

Moved to multiple liked movies:

liked_movies=[
    'Batman',
    'Batman Returns',
    'Star Wars'
]

Created average preference profile:

content_score += similarity[idx]


---

Step 4

Discovered profile dilution problem:

Mixed movies:

Batman
Titanic
Toy Story

↓

Action
Drama
Comedy
Romance
Animation

↓

Very broad profile


---

Step 5

Improved user profile using ratings:

From:

content_score += similarity[idx]

To:

content_score += (
    similarity[idx] * rating
)

Meaning:

Rating = importance weight


---

Step 6

Improved weighted averaging:

From:

content_score /= len(
    liked_movies
)

To:

content_score /= total_rating

Meaning:

Movie rated 5
→ stronger influence

Movie rated 4
→ smaller influence


---

Step 7

Normalized scores before combining:

Problem:

Collaborative:
1–5

Content:
0–1

Fix:

MinMaxScaler()


---

Step 8

Final hybrid formula:

final_score=
0.7*collab_score
+
0.3*content_score

Experimented with:

0.7 / 0.3
0.5 / 0.5
0.1 / 0.9

Observed:

Higher content weight:

More genre-based movies

Higher collaborative weight:

More behavior-based movies


---

🧠 Major concepts learned today

1. Hybrid recommendation systems


2. User profile creation


3. Weighted user preferences


4. Score normalization


5. Weight tuning


6. Signal engineering


7. Recommendation quality debugging


8. Why combining models is more than simply adding outputs




---

Current project status

Completed:

Day 1 → Day 8

Next likely focus:

🚀 Day 9

Production direction:
FastAPI + serving recommendations through APIs

You’ve moved beyond isolated algorithms and into building a recommendation pipeline.

